# 알약 누끼 추출

train_56_45_merged_coco.zip의 aihub_45_fill 이미지에서
SAM으로 알약 누끼를 추출하여 pills/ 폴더 하위에 클래스별로 폴더 생성 후 저장합니다.

## 셀 1. SAM 설치 및 체크포인트

In [ ]:
!pip install -q segment-anything
import os
SAM_CKPT_PATH = "/content/sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CKPT_PATH):
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth \
        -O {SAM_CKPT_PATH}
    print("SAM 다운로드 완료")
else:
    print("SAM 이미 존재")

## 셀 2. 임포트 및 Drive 마운트

In [ ]:
import os, json, glob, random, zipfile, io
import numpy as np
import cv2
from PIL import Image
from collections import defaultdict, Counter
from google.colab import drive
drive.mount('/content/drive')
print("Drive 마운트 완료")
from segment_anything import sam_model_registry, SamPredictor
print("SAM 임포트 완료")

## 셀 3. 경로 설정

In [ ]:
DRIVE_BASE   = "/content/drive/MyDrive"
PILLS_X_DIR  = os.path.join(DRIVE_BASE, "pills")
PILLS_N_DIR  = os.path.join(DRIVE_BASE, "pills_n18")
BG_DIR       = os.path.join(DRIVE_BASE, "backgrounds")
DATASET_DIR  = "/content/dataset"
ZIP2_PATH    = os.path.join(DRIVE_BASE, "train_56_45_merged_coco_20260703.zip")
CLEANED2_PATH= os.path.join(DRIVE_BASE, "train_56_45_cleaned.json")
CLEANED_N_PATH= os.path.join(DRIVE_BASE, "hidden_n18_cleaned.json")
ZIP_N_PATH   = os.path.join(DRIVE_BASE, "hidden_n18_aihub_train_import_20260703.zip")
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"

CLUSTER = {
    "top_left"   : (230, 330, 30, 40), "top_center" : (465, 330, 30, 40),
    "top_right"  : (700, 330, 35, 40), "bot_left"   : (245, 890, 35, 60),
    "bot_center" : (465, 1000, 30, 40),"bot_right"  : (715, 840, 40, 50),
}
LAYOUT_3 = [["top_left","top_right","bot_center"],["top_center","bot_left","bot_right"]]
LAYOUT_4 = [["top_left","top_right","bot_left","bot_right"]]
SHADOW_OFFSET=(12,12); SHADOW_BLUR=18; SHADOW_OPACITY=0.45; BLEND_FEATHER=3
TARGET_COUNT=35; ROTATE_RANGE=15; RESIZE_JITTER=0.10
OVERLAP_MARGIN=10; MAX_POS_RETRIES=5; SCALE_FACTOR=0.85

for name, p in [("PILLS_X_DIR",PILLS_X_DIR),("BG_DIR",BG_DIR),
                ("ZIP2_PATH",ZIP2_PATH),("CLEANED2_PATH",CLEANED2_PATH)]:
    print(f"{name:20s}: {'OK' if os.path.exists(p) else 'MISSING'}")

## 셀 4. cleaned2 로드 및 aihub_45_fill 필터링

In [ ]:
# image_id 불일치 확인
ann_image_ids = {ann["image_id"] for ann in cleaned2["annotations"]}
img_ids = {img["id"] for img in cleaned2["images"]}

missing = ann_image_ids - img_ids
print(f"annotations에는 있지만 images에 없는 image_id: {missing}")
print(f"images 수: {len(img_ids)}")
print(f"annotations 수: {len(cleaned2['annotations'])}")

In [ ]:
# 방법 A: 불일치 annotation 제거 (권장)
valid_img_ids = {img["id"] for img in cleaned2["images"]}
cleaned2["annotations"] = [ann for ann in cleaned2["annotations"]
                           if ann["image_id"] in valid_img_ids]
print(f"정리 후 어노테이션: {len(cleaned2['annotations'])}개")

# 저장
with open(CLEANED2_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaned2, f, ensure_ascii=False)
print("저장 완료")

In [ ]:
with open(CLEANED2_PATH, encoding="utf-8") as f:
    cleaned2 = json.load(f)

id2img2  = {img["id"]: img for img in cleaned2["images"]}
id2name2 = {cat["id"]: cat["name"] for cat in cleaned2["categories"]}

# aihub_45_fill만 필터링
aihub_anns = [ann for ann in cleaned2["annotations"]
              if id2img2[ann["image_id"]].get("source_dataset") == "aihub_45_fill"]

print(f"전체 어노테이션: {len(cleaned2['annotations'])}개")
print(f"aihub_45_fill : {len(aihub_anns)}개")

cnt = Counter(id2name2[a["category_id"]] for a in aihub_anns)
print(f"클래스 수: {len(cnt)} / min={min(cnt.values())} / max={max(cnt.values())}")

In [ ]:
# aihub_45_fill에 있는 클래스 확인
aihub_cats = {id2name2[a["category_id"]] for a in aihub_anns}
all_cats   = {cat["name"] for cat in cleaned2["categories"]}

print("aihub_45_fill에 없는 클래스:")
for name in sorted(all_cats - aihub_cats):
    print(f"  {name}")

## 셀 5. SAM 함수 정의

In [7]:
def load_sam(checkpoint=SAM_CHECKPOINT, model_type="vit_h", device="cuda"):
    sam = sam_model_registry[model_type](checkpoint=checkpoint)
    sam.to(device=device)
    print(f"SAM 로드 완료 | {model_type} | {device}")
    return SamPredictor(sam)

def segment_pill(predictor, image_bgr, bbox_xywh):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)
    x, y, w, h = [float(v) for v in bbox_xywh]
    box = np.array([x, y, x+w, y+h])
    masks, scores, _ = predictor.predict(box=box[None,:], multimask_output=True)
    return masks[int(np.argmax(scores))]

def crop_pill_rgba(image_bgr, mask, bbox_xywh, padding=5):
    H, W = image_bgr.shape[:2]
    x, y, w, h = [int(v) for v in bbox_xywh]
    x1,y1 = max(0,x-padding), max(0,y-padding)
    x2,y2 = min(W,x+w+padding), min(H,y+h+padding)
    crop_bgr  = image_bgr[y1:y2,x1:x2].copy()
    crop_mask = mask[y1:y2,x1:x2].astype(np.uint8)*255
    k = BLEND_FEATHER*2+1
    crop_mask = cv2.GaussianBlur(crop_mask,(k,k),BLEND_FEATHER)
    rgba = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2BGRA)
    rgba[:,:,3] = crop_mask
    return rgba

## 셀 6. 누끼 추출 함수 및 실행

> 약 **15~25분** 소요 (1821장 기준)

In [ ]:
def extract_pills_x(predictor, aihub_anns, id2img2, id2name2,
                    zip_path, output_dir=PILLS_X_DIR, padding=5):
    class_counts = defaultdict(int)
    os.makedirs(output_dir, exist_ok=True)
    total = len(aihub_anns)
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, ann in enumerate(aihub_anns):
            print(f"[{i+1}/{total}]", end="\r")
            fn       = id2img2[ann["image_id"]]["file_name"]  # "train/N01_..."
            cat_name = id2name2[ann["category_id"]]           # dl_idx string
            try:
                img_bytes = z.read(f"train_56_45_merged_coco/images/{fn}")
                img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                image_bgr = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
                mask      = segment_pill(predictor, image_bgr, ann["bbox"])
                rgba      = crop_pill_rgba(image_bgr, mask, ann["bbox"], padding)
                cls_dir   = os.path.join(output_dir, f"class_{cat_name}")
                os.makedirs(cls_dir, exist_ok=True)
                idx       = class_counts[cat_name]
                cv2.imwrite(os.path.join(cls_dir, f"pill_{idx:04d}.png"), rgba)
                class_counts[cat_name] += 1
            except Exception as e:
                print(f"\n[ERROR] {fn}: {e}")
    print(f"\n추출 완료: 총 {sum(class_counts.values())}개")
    print(f"클래스별: min={min(class_counts.values())} / max={max(class_counts.values())}")
    return dict(class_counts)

predictor = load_sam(SAM_CHECKPOINT)
x_class_counts = extract_pills_x(predictor, aihub_anns, id2img2, id2name2, ZIP2_PATH)

## 셀 7. 결과 확인

In [ ]:
print(f"pills/ 폴더 클래스 수: {len(os.listdir(PILLS_X_DIR))}개")
print(f"{'class':>10} {'nukkis':>8}")
print("-"*22)
for cls_dir in sorted(os.listdir(PILLS_X_DIR)):
    if not cls_dir.startswith("class_"): continue
    cnt = len(os.listdir(os.path.join(PILLS_X_DIR, cls_dir)))
    flag = " ⚠️" if cnt < 35 else ""
    print(f"  {cls_dir:>15} {cnt:>6}{flag}")